In [6]:
import numpy as np
import heapq
import pandas as pd

In [7]:
# -----------------------------
# PARAMETERS
# -----------------------------

SIM_TIME = 1000         
DRIVER_RATE = 3
RIDER_RATE = 30
PATIENCE_RATE = 5

SPEED = 20
AREA_SIZE = 20

BASE_FARE = 3
FARE_PER_MILE = 2
PETROL_COST_PER_MILE = 0.20


In [8]:


# -----------------------------
# DRIVER CLASS
# -----------------------------

class Driver:
    def __init__(self, driver_id, location, online_until):
        self.id = driver_id
        self.location = location
        self.online_until = online_until
        self.earnings = 0
        self.status = "idle"
        self.busy_time = 0



In [9]:

# -----------------------------
# RIDER CLASS
# -----------------------------

class Rider:
    def __init__(self, rider_id, origin, destination, arrival_time, abandon_time):
        self.id = rider_id
        self.origin = origin
        self.destination = destination
        self.arrival_time = arrival_time
        self.abandon_time = abandon_time
        self.matched = False



In [10]:

# -----------------------------
# SIMULATION CLASS
# -----------------------------

class Simulation:

    def __init__(self):
        self.current_time = 0
        self.event_list = []

        self.idle_drivers = {}
        self.busy_drivers = {}
        self.waiting_riders = {}

        self.driver_counter = 0
        self.rider_counter = 0

        # statistics
        self.total_riders = 0
        self.abandoned_riders = 0
        self.total_wait_time = 0

        # logs
        self.driver_log = []
        self.rider_log = []

    # -----------------------------
    # RANDOM HELPERS
    # -----------------------------

    def exp_time(self, rate):
        return np.random.exponential(1/rate)

    def uniform_location(self):
        return np.random.uniform(0, AREA_SIZE, size=2)

    def distance(self, a, b):
        return np.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

    def travel_time(self, dist):
        mu = dist / SPEED
        return np.random.uniform(0.8*mu, 1.2*mu)

    # -----------------------------
    # EVENT SCHEDULER
    # -----------------------------

    def schedule_event(self, time, event_type, data=None):
        heapq.heappush(self.event_list, (time, event_type, data))

    # -----------------------------
    # MATCHING
    # -----------------------------

    def try_match(self):
        if not self.idle_drivers or not self.waiting_riders:
            return

        driver_id = next(iter(self.idle_drivers))
        driver = self.idle_drivers[driver_id]

        rider_id = min(
            self.waiting_riders,
            key=lambda r: self.distance(driver.location,
                                        self.waiting_riders[r].origin)
        )

        rider = self.waiting_riders[rider_id]
        rider.matched = True

        wait_time = self.current_time - rider.arrival_time
        self.total_wait_time += wait_time

        pickup_dist = self.distance(driver.location, rider.origin)
        trip_dist = self.distance(rider.origin, rider.destination)

        pickup_time = self.travel_time(pickup_dist)
        trip_time = self.travel_time(trip_dist)

        total_trip_time = pickup_time + trip_time

        payment = BASE_FARE + FARE_PER_MILE * trip_dist
        petrol = PETROL_COST_PER_MILE * (pickup_dist + trip_dist)
        earnings = payment - petrol

        driver.status = "busy"
        driver.busy_time += total_trip_time

        self.busy_drivers[driver_id] = driver

        del self.idle_drivers[driver_id]
        del self.waiting_riders[rider_id]

        dropoff_time = self.current_time + total_trip_time

        self.schedule_event(dropoff_time,
                            "DROPOFF",
                            (driver_id, rider.destination, earnings))

    # -----------------------------
    # EVENT HANDLERS
    # -----------------------------

    def handle_driver_arrival(self):
        self.driver_counter += 1
        driver_id = self.driver_counter

        location = self.uniform_location()
        online_duration = np.random.uniform(5, 8)
        online_until = self.current_time + online_duration

        driver = Driver(driver_id, location, online_until)
        self.idle_drivers[driver_id] = driver

        self.driver_log.append({
            "driver_id": driver_id,
            "arrival_time": self.current_time,
            "online_until": online_until,
            "earnings": 0,
            "busy_time": 0
        })

        self.schedule_event(online_until, "DRIVER_OFFLINE", driver_id)
        self.schedule_event(self.current_time + self.exp_time(DRIVER_RATE),
                            "DRIVER_ARRIVAL")

        self.try_match()

    def handle_rider_arrival(self):
        self.rider_counter += 1
        rider_id = self.rider_counter
        self.total_riders += 1

        origin = self.uniform_location()
        destination = self.uniform_location()
        abandon_time = self.current_time + self.exp_time(PATIENCE_RATE)

        rider = Rider(rider_id, origin, destination,
                      self.current_time, abandon_time)

        self.waiting_riders[rider_id] = rider

        self.rider_log.append({
            "rider_id": rider_id,
            "arrival_time": self.current_time
        })

        self.schedule_event(abandon_time, "RIDER_ABANDON", rider_id)
        self.schedule_event(self.current_time + self.exp_time(RIDER_RATE),
                            "RIDER_ARRIVAL")

        self.try_match()

    def handle_dropoff(self, data):
        driver_id, new_location, earnings = data

        driver = self.busy_drivers[driver_id]
        driver.earnings += earnings
        driver.location = new_location
        driver.status = "idle"

        # update driver log
        for d in self.driver_log:
            if d["driver_id"] == driver_id:
                d["earnings"] += earnings
                d["busy_time"] = driver.busy_time

        del self.busy_drivers[driver_id]

        if self.current_time < driver.online_until:
            self.idle_drivers[driver_id] = driver

        self.try_match()

    # -----------------------------
    # MAIN RUN
    # -----------------------------

    def run(self):
        self.schedule_event(0, "DRIVER_ARRIVAL")
        self.schedule_event(0, "RIDER_ARRIVAL")

        while self.event_list and self.current_time < SIM_TIME:
            self.current_time, event_type, data = heapq.heappop(self.event_list)

            if event_type == "DRIVER_ARRIVAL":
                self.handle_driver_arrival()

            elif event_type == "RIDER_ARRIVAL":
                self.handle_rider_arrival()

            elif event_type == "DROPOFF":
                self.handle_dropoff(data)

            elif event_type == "RIDER_ABANDON":
                if data in self.waiting_riders:
                    self.abandoned_riders += 1
                    del self.waiting_riders[data]

            elif event_type == "DRIVER_OFFLINE":
                if data in self.idle_drivers:
                    del self.idle_drivers[data]

        self.report()

    # -----------------------------
    # PERFORMANCE REPORT
    # -----------------------------

    def report(self):

        abandonment_rate = self.abandoned_riders / self.total_riders

        matched_riders = self.total_riders - self.abandoned_riders
        avg_wait = self.total_wait_time / matched_riders if matched_riders > 0 else 0

        driver_df = pd.DataFrame(self.driver_log)

        driver_df["online_duration"] = \
            driver_df["online_until"] - driver_df["arrival_time"]

        driver_df["earnings_per_hour"] = \
            driver_df["earnings"] / driver_df["online_duration"]

        avg_earnings_per_hour = driver_df["earnings_per_hour"].mean()

        mean_earnings = driver_df["earnings"].mean()
        std_earnings = driver_df["earnings"].std()

        fairness_cv = std_earnings / mean_earnings if mean_earnings > 0 else 0

        driver_df["idle_time"] = \
            driver_df["online_duration"] - driver_df["busy_time"]

        avg_idle_time = driver_df["idle_time"].mean()

        print("\n===== RIDER SATISFACTION =====")
        print("Total Riders:", self.total_riders)
        print("Abandonment Rate:", round(abandonment_rate,4))
        print("Average Waiting Time:", round(avg_wait,4))

        print("\n===== DRIVER SATISFACTION =====")
        print("Average Earnings per Hour:", round(avg_earnings_per_hour,4))
        print("Fairness (CV of earnings):", round(fairness_cv,4))
        print("Average Idle Time:", round(avg_idle_time,4))


# -----------------------------
# RUN SIMULATION
# -----------------------------

sim = Simulation()
sim.run()


===== RIDER SATISFACTION =====
Total Riders: 29966
Abandonment Rate: 0.2803
Average Waiting Time: 0.0432

===== DRIVER SATISFACTION =====
Average Earnings per Hour: 21.5172
Fairness (CV of earnings): 0.2025
Average Idle Time: -0.2039
